# Streaming Tokens to the User: SSE, WebSockets, and the Wait the Reader Feels

The runnable code of [Streaming Tokens to the User: SSE, WebSockets, and the Wait the Reader Feels](https://residual-stream.lvtan71.workers.dev/posts/streaming-tokens-sse-websockets/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/streaming-tokens-sse-websockets.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 2 min end to end.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "streaming-tokens-sse-websockets"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/streaming-tokens-sse-websockets/client.py', 'shared/streaming-tokens-sse-websockets/detokenizer.py', 'shared/streaming-tokens-sse-websockets/lm.py', 'shared/decoding-from-logits-to-text/sampling.py', 'data/prompts.json']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## A streaming server, from the standard library

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/streaming-tokens-sse-websockets/#a-streaming-server-from-the-standard-library)

In [ ]:
%%writefile server.py
import asyncio
import base64
import hashlib
import json
import sys
import threading
import time
import torch
from lm import model, tok, DEVICE
from sampling import choose
from detokenizer import Detokenizer

# One model, three ways to deliver its tokens, on Python's standard library:
#   POST /complete  the whole answer as one JSON response, at the end;
#   POST /stream    server-sent events, one event per piece of text;
#   GET  /ws        a WebSocket, which also lets the client say "stop";
#   GET  /log       what was generated for each request so far.
# Generation runs in a thread and checks a stop flag before every token.
# With --naive the server never notices that a client has left.
NAIVE = "--naive" in sys.argv
LOG = []                                                  # (endpoint, tokens generated, client left?)

def generate(prompt, n, out, stop, loop):
    """Sample up to n tokens (top-p 0.9, seeded), pushing text pieces to the
    asyncio queue `out`; a None marks the end."""
    m, ids = model(), torch.tensor([tok(prompt).input_ids], device=DEVICE)
    g = torch.Generator(device=DEVICE).manual_seed(0)
    d, history, produced = Detokenizer(), [], 0
    with torch.no_grad():
        result = m(ids, use_cache=True)
        for _ in range(n):
            if stop.is_set():
                break
            token = choose(result.logits[0, -1], history, g, p=0.9)
            history.append(token); produced += 1
            if token == tok.eos_token_id:
                break
            loop.call_soon_threadsafe(out.put_nowait, d.push(token))
            result = m(torch.tensor([[token]], device=DEVICE), past_key_values=result.past_key_values, use_cache=True)
    loop.call_soon_threadsafe(out.put_nowait, None)
    return produced

async def start(prompt, n):
    out, stop, loop = asyncio.Queue(), threading.Event(), asyncio.get_running_loop()
    job = loop.run_in_executor(None, generate, prompt, n, out, stop, loop)
    return out, stop, job

async def handle(reader, writer):
    head = (await reader.readuntil(b"\r\n\r\n")).decode()
    method, path, _ = head.split("\r\n")[0].split(" ")
    headers = {k.lower(): v for k, v in (l.split(": ", 1) for l in head.split("\r\n")[1:] if ": " in l)}
    if path == "/ws":
        return await websocket(reader, writer, headers)
    if path == "/log":                                    # what the server did, for the experiments
        payload = json.dumps(LOG).encode()
        writer.write(b"HTTP/1.1 200 OK\r\nContent-Length: %d\r\nConnection: close\r\n\r\n" % len(payload) + payload)
        return writer.close()
    body = json.loads(await reader.readexactly(int(headers["content-length"])))
    out, stop, job = await start(body["prompt"], body.get("max_tokens", 100))
    left = False
    if path == "/complete":
        pieces = []
        while (piece := await out.get()) is not None:
            pieces.append(piece)
        payload = json.dumps({"text": "".join(pieces)}).encode()
        writer.write(b"HTTP/1.1 200 OK\r\nContent-Type: application/json\r\n"
                     b"Content-Length: %d\r\nConnection: close\r\n\r\n" % len(payload) + payload)
    else:
        writer.write(b"HTTP/1.1 200 OK\r\nContent-Type: text/event-stream\r\nCache-Control: no-cache\r\n"
                     b"X-Accel-Buffering: no\r\nConnection: close\r\n\r\n")
        if not NAIVE:                                     # the client closing its side means it's gone
            asyncio.create_task(watch(reader, stop))
        while (piece := await out.get()) is not None:
            if piece:
                writer.write(b"data: " + json.dumps({"text": piece}).encode() + b"\n\n")
                try:
                    await writer.drain()
                except ConnectionError:
                    left = True
                    if not NAIVE:
                        stop.set()
        writer.write(b"data: [DONE]\n\n")
        left = left or stop.is_set()
    LOG.append((path, await job, left))
    print(f"{path}: generated {LOG[-1][1]} tokens, client left early: {left}", flush=True)
    try:
        await writer.drain(); writer.close()
    except ConnectionError:
        pass

async def watch(reader, stop):
    await reader.read()                                   # returns b"" when the client disconnects
    stop.set()

# --- WebSocket (RFC 6455), by hand: an HTTP upgrade, then framed messages.
async def websocket(reader, writer, headers):
    accept = base64.b64encode(hashlib.sha1((headers["sec-websocket-key"] +
                              "258EAFA5-E914-47DA-95CA-C5AB0DC85B11").encode()).digest())
    writer.write(b"HTTP/1.1 101 Switching Protocols\r\nUpgrade: websocket\r\nConnection: Upgrade\r\n"
                 b"Sec-WebSocket-Accept: " + accept + b"\r\n\r\n")
    request = json.loads(await read_frame(reader))
    out, stop, job = await start(request["prompt"], request.get("max_tokens", 100))
    async def listen():                                   # the client can talk while the server streams
        while True:
            message = await read_frame(reader)
            if message is None or json.loads(message).get("stop"):
                stop.set(); return
    listener = asyncio.create_task(listen())
    while (piece := await out.get()) is not None:
        if piece:
            writer.write(frame(json.dumps({"text": piece}).encode()))
    writer.write(frame(json.dumps({"done": True}).encode()))
    await writer.drain()
    LOG.append(("/ws", await job, stop.is_set()))
    print(f"/ws: generated {LOG[-1][1]} tokens, client said stop: {stop.is_set()}", flush=True)
    listener.cancel(); writer.close()

def frame(payload, opcode=0x1):
    """A server-to-client frame: FIN + opcode, then the length, then the data."""
    n = len(payload)
    head = bytes([0x80 | opcode]) + (bytes([n]) if n < 126 else bytes([126]) + n.to_bytes(2, "big"))
    return head + payload

async def read_frame(reader):
    """A client-to-server frame; clients must mask their data with 4 random bytes."""
    try:
        b0, b1 = await reader.readexactly(2)
    except asyncio.IncompleteReadError:
        return None
    n = b1 & 0x7F
    if n == 126:
        n = int.from_bytes(await reader.readexactly(2), "big")
    mask, data = await reader.readexactly(4), await reader.readexactly(n)
    if b0 & 0x0F == 0x8:                                  # a close frame
        return None
    return bytes(b ^ mask[i % 4] for i, b in enumerate(data)).decode()

async def main():
    model()                                               # load the weights before taking requests
    server = await asyncio.start_server(handle, "127.0.0.1", 8765)
    print("listening on 8765" + (" (naive)" if NAIVE else ""), flush=True)
    async with server:
        await server.serve_forever()

if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
# The post ran the server in a terminal; here it runs in the background.
import socket, time
def start(*args, port):
    p = subprocess.Popen([sys.executable, *args])
    while True:                                  # until it listens (the model loads first)
        if p.poll() is not None:
            raise RuntimeError(f"{args[0]} exited")
        try:
            socket.create_connection(("127.0.0.1", port)).close()
            return p
        except OSError:
            time.sleep(1)

server = start("server.py", port=8765)

## A trap in sending text one token at a time

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/streaming-tokens-sse-websockets/#a-trap-in-sending-text-one-token-at-a-time)

In [ ]:
from lm import tok

# Streaming text one token at a time has a trap: a token is a piece of a
# byte sequence, not of a string, and one character can span two tokens.
# Decoding each token alone then prints broken characters. The fix is
# incremental: decode everything so far, and emit only the new text once it
# no longer ends in a half-finished character.
class Detokenizer:
    def __init__(self):
        self.ids, self.sent = [], ""
    def push(self, token_id):
        """Add a token; return the new text that is safe to send (may be '')."""
        self.ids.append(token_id)
        text = tok.decode(self.ids)
        if text.endswith("�"):            # the last character isn't complete yet
            return ""
        new, self.sent = text[len(self.sent):], text
        return new

In [ ]:
from lm import tok
from detokenizer import Detokenizer

# A character split across tokens, decoded one token at a time and
# incrementally.
text = "Con vẹt 🦜 nói: mô hình sinh văn bản từng token một 🫠"
ids = tok(text).input_ids
naive = [tok.decode([i]) for i in ids]
print(f"{len(ids)} tokens; decoded one by one:")
print("  " + " | ".join(naive))

d = Detokenizer()
pieces = [d.push(i) for i in ids]
print("incremental:")
print("  " + " | ".join(pieces))
print(f"broken pieces one by one: {sum('�' in p for p in naive)}, incremental: "
      f"{sum('�' in p for p in pieces)}; joined text equal to the original: {''.join(pieces) == text}")

The post printed:

```text
20 tokens; decoded one by one:
  Con |  v | ẹ | t |  � | � | � |  nói | : |  mô |  hình |  sinh |  văn |  bản |  từng |  token |  một |  � | � | �
incremental:
  Con |  v | ẹ | t |  |  |  🦜 |  nói | : |  mô |  hình |  sinh |  văn |  bản |  từng |  token |  một |  |  |  🫠
broken pieces one by one: 6, incremental: 0; joined text equal to the original: True
```

## Measured: the first text, and the bytes

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/streaming-tokens-sse-websockets/#measured-the-first-text-and-the-bytes)

In [ ]:
%%writefile client.py
import asyncio
import base64
import json
import os
import sys
import time

# Measuring the server from the outside, over raw sockets, so every byte on
# the wire is counted: when the first text arrives, when the last does, and
# how many bytes it took.
PORT = int(os.environ.get("PORT", 8765))

async def post(path, prompt, n, stop_after=None):
    """POST a prompt; returns (seconds to first text, seconds to the end,
    bytes received, pieces of text). Closes early after `stop_after` events."""
    r, w = await asyncio.open_connection("127.0.0.1", PORT)
    body = json.dumps({"prompt": prompt, "max_tokens": n}).encode()
    t0 = time.perf_counter()
    w.write(b"POST %s HTTP/1.1\r\nHost: x\r\nContent-Length: %d\r\n\r\n" % (path.encode(), len(body)) + body)
    first, received, pieces, buffer = None, 0, [], b""
    head_done = False
    while chunk := await r.read(65536):
        received += len(chunk); buffer += chunk
        if not head_done and b"\r\n\r\n" in buffer:
            head_done, buffer = True, buffer.split(b"\r\n\r\n", 1)[1]
        if head_done and path == "/stream":
            while b"\n\n" in buffer:
                event, buffer = buffer.split(b"\n\n", 1)
                data = event[len(b"data: "):]
                if data != b"[DONE]":
                    first = first or time.perf_counter() - t0
                    pieces.append(json.loads(data)["text"])
            if stop_after and len(pieces) >= stop_after:
                w.close(); await asyncio.sleep(1.0)       # leave, then give the server a moment to notice
                return first, time.perf_counter() - t0, received, pieces
    if path == "/complete":
        first = time.perf_counter() - t0
        pieces = [json.loads(buffer)["text"]]
    return first, time.perf_counter() - t0, received, pieces

def ws_frame(payload):
    """A client-to-server text frame, masked as RFC 6455 requires."""
    mask, n = os.urandom(4), len(payload)
    size = bytes([0x80 | n]) if n < 126 else bytes([0x80 | 126]) + n.to_bytes(2, "big")
    return bytes([0x81]) + size + mask + bytes(b ^ mask[i % 4] for i, b in enumerate(payload))

async def websocket(prompt, n, stop_after=None):
    r, w = await asyncio.open_connection("127.0.0.1", PORT)
    key = base64.b64encode(os.urandom(16))
    t0 = time.perf_counter()
    w.write(b"GET /ws HTTP/1.1\r\nHost: x\r\nUpgrade: websocket\r\nConnection: Upgrade\r\n"
            b"Sec-WebSocket-Version: 13\r\nSec-WebSocket-Key: " + key + b"\r\n\r\n")
    received = len(await r.readuntil(b"\r\n\r\n"))
    w.write(ws_frame(json.dumps({"prompt": prompt, "max_tokens": n}).encode()))
    first, pieces = None, []
    while True:
        b0, b1 = await r.readexactly(2)
        size = b1 if b1 < 126 else int.from_bytes(await r.readexactly(2), "big")
        message = json.loads(await r.readexactly(size))
        received += 2 + (2 if b1 >= 126 else 0) + size
        if message.get("done"):
            break
        first = first or time.perf_counter() - t0
        pieces.append(message["text"])
        if stop_after and len(pieces) == stop_after:
            w.write(ws_frame(b'{"stop": true}'))           # same connection, other direction
    w.close()
    return first, time.perf_counter() - t0, received, pieces

In [ ]:
# client.py is imported by the cells below.

In [ ]:
import asyncio
import json
from client import post, websocket

# Five prompts from the "Decoding" set, 100 tokens each, delivered three ways.
prompts = [p["prompt"] for p in json.load(open("prompts.json"))][:5]

async def main():
    rows = {"/complete": [], "/stream": [], "/ws": []}
    for prompt in prompts:
        rows["/complete"].append(await post("/complete", prompt, 100))
        rows["/stream"].append(await post("/stream", prompt, 100))
        rows["/ws"].append(await websocket(prompt, 100))
    for name, rs in rows.items():
        first = sum(r[0] for r in rs) / len(rs); total = sum(r[1] for r in rs) / len(rs)
        text = ["".join(r[3]) for r in rs]
        received = sum(r[2] for r in rs); chars = sum(len(t.encode()) for t in text)
        print(f"{name:10s} first text after {first:5.2f} s, all of it after {total:5.2f} s; "
              f"{received / 5:6.0f} bytes on the wire per answer, for {chars / 5:4.0f} bytes of text "
              f"({received / chars:.1f}x)")
    same = all("".join(a[3]) == "".join(b[3]) == "".join(c[3]) for a, b, c in zip(*rows.values()))
    print(f"the same text all three ways: {same}")

await main()

The post printed:

```text
/complete  first text after  4.68 s, all of it after  4.68 s;    596 bytes on the wire per answer, for  488 bytes of text (1.2x)
/stream    first text after  0.13 s, all of it after  4.75 s;   2626 bytes on the wire per answer, for  488 bytes of text (5.4x)
/ws        first text after  0.13 s, all of it after  5.12 s;   2038 bytes on the wire per answer, for  488 bytes of text (4.2x)
the same text all three ways: True
```

In [ ]:
import json

# What one streamed piece of text, "Two", costs on the wire in different
# formats, as compact JSON. The Anthropic event is the documented example's
# shape; the OpenAI chunk has the fields of the example in OpenAI's cookbook
# (current chunks also carry a padding field, below).
text = "Two"
ours = b"data: " + json.dumps({"text": text}).encode() + b"\n\n"
anthropic = (b"event: content_block_delta\ndata: " + json.dumps(
    {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": text}},
    separators=(",", ":")).encode() + b"\n\n")
openai = b"data: " + json.dumps(
    {"id": "chatcmpl-9lMgfRSWPHcw51s6wxKT1YEO2CKpd", "object": "chat.completion.chunk", "created": 1721075653,
     "model": "gpt-july-test", "system_fingerprint": "fp_e9b8ed65d2",
     "choices": [{"index": 0, "delta": {"content": text}, "logprobs": None, "finish_reason": None}]},
    separators=(",", ":")).encode() + b"\n\n"
payload = json.dumps({"text": text}).encode()
websocket = bytes([0x81, len(payload)]) + payload            # a server-to-client frame: 2 bytes of header
for name, b in (("this post's SSE event", ours), ("Anthropic content_block_delta", anthropic),
                ("OpenAI chat.completion.chunk", openai), ("this post's WebSocket frame", websocket)):
    print(f"{name:31s} {len(b):4d} bytes for {len(text.encode())} bytes of text")

The post printed:

```text
this post's SSE event             23 bytes for 3 bytes of text
Anthropic content_block_delta    118 bytes for 3 bytes of text
OpenAI chat.completion.chunk     257 bytes for 3 bytes of text
this post's WebSocket frame       17 bytes for 3 bytes of text
```

## What the reader feels

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/streaming-tokens-sse-websockets/#what-the-reader-feels)

In [ ]:
import json
from lm import tok

# Is generation faster than reading? Tokens per word on the blog's own
# prose (the 20 real continuations from "Decoding"), against a reader's speed: 238
# words a minute for non-fiction, the average across studies (Brysbaert,
# 2019). Generation speed is from measure.py: 100 tokens in about 4.7 s.
texts = [p["human"] for p in json.load(open("prompts.json"))]
tokens = sum(len(tok(t).input_ids) for t in texts)
words = sum(len(t.split()) for t in texts)
per_word = tokens / words
reading = 238 / 60 * per_word                              # tokens a second a reader gets through
generating = 100 / 4.7
print(f"{per_word:.2f} tokens per word; a reader at 238 words a minute takes in {reading:.1f} tokens a second; "
      f"this laptop generates {generating:.0f}, {generating / reading:.1f} times as fast")
for n_words in (50, 300):
    n = n_words * per_word
    print(f"a {n_words}-word answer ({n:.0f} tokens): {n / generating:.1f} s to generate, {n_words / 238 * 60:.0f} s to read; "
          f"without streaming the reader waits {n / generating:.1f} s before the first word, with it about 0.13 s")

The post printed:

```text
1.38 tokens per word; a reader at 238 words a minute takes in 5.5 tokens a second; this laptop generates 21, 3.9 times as fast
a 50-word answer (69 tokens): 3.2 s to generate, 13 s to read; without streaming the reader waits 3.2 s before the first word, with it about 0.13 s
a 300-word answer (415 tokens): 19.5 s to generate, 76 s to read; without streaming the reader waits 19.5 s before the first word, with it about 0.13 s
```

## When the user leaves

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/streaming-tokens-sse-websockets/#when-the-user-leaves)

In [ ]:
import asyncio
import json
import os
import client
from client import post, websocket

# Three experiments against the running server (and, for the last, through
# the proxy on port 8766):
#   leave:   the client goes away after 10 pieces of text. Does the server stop?
#   stop:    over a WebSocket, the client asks it to stop after 10 pieces.
#   proxy:   when does the first text arrive through a buffering proxy?
PROMPT = json.load(open("prompts.json"))[3]["prompt"]

async def server_log():
    r, w = await asyncio.open_connection("127.0.0.1", 8765)
    w.write(b"GET /log HTTP/1.1\r\nHost: x\r\n\r\n")
    return json.loads((await r.read()).split(b"\r\n\r\n", 1)[1])

async def main(which):
    if which == "leave":
        await post("/stream", PROMPT, 100, stop_after=10)
        await asyncio.sleep(5)                            # let a server that didn't notice finish
        path, generated, left = (await server_log())[-1]
        print(f"SSE client left after 10 pieces: the server generated {generated} tokens in all")
    if which == "stop":
        await websocket(PROMPT, 100, stop_after=10)
        path, generated, stopped = (await server_log())[-1]
        print(f"WebSocket client said stop after 10 pieces: the server generated {generated} tokens in all")
    if which == "proxy":
        client.PORT = 8766
        first, total, received, pieces = await post("/stream", PROMPT, 100)
        print(f"through the proxy: first text after {first:.2f} s, the end after {total:.2f} s, "
              f"{len(pieces)} pieces, {received:,} bytes")

# The post ran main() once per experiment; the cells below do the same.

In [ ]:
await main("leave")

The post printed:

```text
SSE client left after 10 pieces: the server generated 10 tokens in all
```

In [ ]:
# The same experiment against the server started with --naive:
server.terminate(); server.wait()
server = start("server.py", "--naive", port=8765)
await main("leave")

The post printed:

```text
SSE client left after 10 pieces: the server generated 100 tokens in all
```

In [ ]:
server.terminate(); server.wait()
server = start("server.py", port=8765)
await main("stop")

The post printed:

```text
WebSocket client said stop after 10 pieces: the server generated 10 tokens in all
```

## The proxy that undoes it all

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/streaming-tokens-sse-websockets/#the-proxy-that-undoes-it-all)

In [ ]:
%%writefile proxy.py
import asyncio
import sys

# A stand-in for a reverse proxy between the user and the server, the kind
# every deployment has (nginx, a load balancer, a CDN). This one buffers the
# worst way a hop can: it collects the upstream response and forwards it
# only in blocks of 8 KB, or when the upstream finishes.
# With --honor it skips buffering for responses that ask for that with the
# header nginx uses, X-Accel-Buffering: no.
HONOR = "--honor" in sys.argv
BLOCK = 8192

async def pipe_up(reader, writer):
    while data := await reader.read(65536):
        writer.write(data); await writer.drain()

async def handle(client_r, client_w):
    up_r, up_w = await asyncio.open_connection("127.0.0.1", 8765)
    asyncio.create_task(pipe_up(client_r, up_w))
    buffered, head, passthrough = b"", None, False
    while data := await up_r.read(65536):
        if head is None:
            head = data.split(b"\r\n\r\n", 1)[0]
            passthrough = HONOR and b"x-accel-buffering: no" in head.lower()
        if passthrough:
            client_w.write(data); await client_w.drain(); continue
        buffered += data
        while len(buffered) >= BLOCK:
            client_w.write(buffered[:BLOCK]); buffered = buffered[BLOCK:]; await client_w.drain()
    client_w.write(buffered); await client_w.drain(); client_w.close()

async def main():
    server = await asyncio.start_server(handle, "127.0.0.1", 8766)
    print("proxy on 8766" + (", honoring X-Accel-Buffering" if HONOR else ""), flush=True)
    async with server:
        await server.serve_forever()

asyncio.run(main())

In [ ]:
proxy = start("proxy.py", port=8766)
await main("proxy")

The post printed:

```text
through the proxy: first text after 4.91 s, the end after 4.91 s, 100 pieces, 2,583 bytes
```

In [ ]:
# The proxy started with --honor:
proxy.terminate(); proxy.wait()
proxy = start("proxy.py", "--honor", port=8766)
await main("proxy")
proxy.terminate(); server.terminate()

The post printed:

```text
through the proxy: first text after 0.20 s, the end after 5.03 s, 100 pieces, 2,583 bytes
```